# Local PII & GIRP Information Classification

**Runs 100% locally. No internet, no Hugging Face Hub, no API calls.** The model weights and all
config load from files in *this* folder, with Hub access hard-disabled.

Built on a 205M-param GLiNER2 model (DeBERTa-v3 backbone). It can:

1. **Identify PII** — names, emails, phones, SSNs, addresses, cards, government IDs… (zero-shot)
2. **Classify text** — sentiment / intent / topic
3. **Classify GIRP sensitivity** — Public → Private → Confidential → Highly Confidential
4. **Scan the columns you choose in a pandas DataFrame** and tag each one's GIRP level

### How to use
1. Run the **Setup** and **Load model** cells once.
2. In **"Scan your columns"**, point `df` at your data and list the `COLUMNS` to scan. Run it.

Tested on **Windows + Python 3.13**, CPU or NVIDIA GPU (e.g. RTX 2050).

In [ ]:
# --- Setup: install packages into the CURRENT Python (no venv/conda) ---
# Uses pip / your internal package mirror. Safe to re-run; skips what's already present.
import sys, subprocess, importlib.util

REQUIRED = ["gliner2", "pandas"]   # torch + transformers come in automatically with gliner2
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print("Installing:", ", ".join(missing), "...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Done.")
else:
    print("All base packages already installed.")
print("Python:", sys.version.split()[0], "|", sys.executable)

### GPU (optional)

Device is auto-selected: **NVIDIA CUDA if available, otherwise CPU** (CPU is still fast here).
To use your RTX 2050, ensure a **CUDA build of PyTorch** is installed from your internal package
mirror; on CUDA the model runs in fp16 (~0.4 GB VRAM, fits a 4 GB card). No action needed for CPU.

In [ ]:
# --- Load the model: LOCAL files only, no network ---
import os
# Belt-and-braces: disable any Hub/telemetry access before importing the libraries.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import pandas as pd

from girp import (GIRP_PII_LABELS, classify_elements, explain, found_labels,
                  detect_and_classify, classify_columns, classify_dataframe_girp,
                  load_local_model, default_model_dir)

print("Model folder:", default_model_dir())
model, DEVICE = load_local_model()          # local files only — no internet/API calls
print(f"Loaded. Device: {DEVICE}")

## Scan your columns

This is the main entry point. Point `df` at your data, list the columns to scan, and run.
`classify_columns` adds, per column, a `<col>_girp_level` and `<col>_girp_elements`, plus an
overall **`girp_level`** = the most sensitive level found across the chosen columns for that row.

In [ ]:
# ============================================================
#  YOUR DATA  — edit `df` and `COLUMNS`
# ============================================================
# Load your own data (uncomment one) ...
# df = pd.read_csv("your_file.csv")
# df = pd.read_excel("your_file.xlsx")

# ... or use this sample to see it work:
df = pd.DataFrame({
    "customer_note": [
        "Call Sarah Lee on 02 9000 0000 about the refund.",
        "The quarterly report shows strong growth this year.",
        "Patient John Citizen, 12 King Street, treated for depression.",
        "My email is hello@example.com for enquiries.",
    ],
    "agent_comment": [
        "Followed up with the customer, all resolved.",
        "Verified applicant tax file number 123 456 789.",
        "Card 4111 1111 1111 1111 kept on file.",
        "No further action needed.",
    ],
})

COLUMNS = ["customer_note", "agent_comment"]    # <-- the columns to scan
# ============================================================

result = classify_columns(model, df, COLUMNS)
result

In [ ]:
# Overall sensitivity of each row (highest level across the scanned columns)
print("Rows by overall GIRP level:")
print(result["girp_level"].value_counts().to_string())

## GIRP classification rules

Each text is classified into the four-tier GIRP scheme by detecting personal-data elements and
applying the GIRP combination rules:

- **Private** — a personal detail *in isolation* (phone, address, birthplace, mother's maiden name, bank account).
- **Confidential** — a **full name + any** of {phone, DOB, address, birthplace, mother's maiden name}; **or**
  a card number (PAN), government identifier (TFN / Medicare / passport / licence), or health information on its own.
- **Highly Confidential** — health / sensitive information **together with** Confidential-level customer PII.

Rules live in `girp.py`; `test_girp.py` validates them. Scope: the personal-information (Customer)
rules derivable from entities in text — document-type rules are out of scope.

In [ ]:
girp_examples = [
    "The annual report says the group serves five million customers.",   # Public
    "Please call me on +61 412 345 678.",                                # Private (pronoun 'me' ignored)
    "Jane Doe was born on 12/05/1990 and lives at 14 King Street, Sydney.",  # Confidential
    "Card 4111 1111 1111 1111; tax file number 123 456 789.",            # Confidential
    "Patient John Citizen, phone 03 9000 0000, diagnosed with Type 2 diabetes.",  # Highly Confidential
]
for t in girp_examples:
    out = detect_and_classify(model, t)
    print(f"  {out['level']:18s} | elements={out['elements']}")
    print(f"  {'':18s} | {t}")

assert detect_and_classify(model, girp_examples[0])["level"] == "Public"
assert detect_and_classify(model, girp_examples[1])["level"] == "Private"
assert detect_and_classify(model, girp_examples[4])["level"] == "Highly Confidential"
print("\n GIRP classification works.")

In [ ]:
# The rule engine is deterministic — these mirror the GIRP rules (full suite in test_girp.py).
rule_checks = [
    (set(),                                            "Public"),
    ({"phone number"},                                 "Private"),
    ({"person", "phone number"},                       "Confidential"),
    ({"credit card number"},                           "Confidential"),
    ({"person", "phone number", "medical condition"},  "Highly Confidential"),
]
for labels, expected in rule_checks:
    assert classify_elements(labels) == expected, f"{sorted(labels)} -> {classify_elements(labels)}"
print("GIRP rule engine matches the GIRP definitions.")

## PII identification (detail)

GLiNER2 is zero-shot: pass plain-English labels and it returns matching spans. Edit `PII_LABELS`
to add types like `"passport number"`, `"IBAN"`, or `"date of birth"`.

In [ ]:
sample = ("John Smith emailed jane.doe@acme.com from +1 (415) 555-0132. "
          "His SSN is 123-45-6789 and he lives at 742 Evergreen Terrace, Springfield. "
          "Paid with card 4111 1111 1111 1111 on 2024-03-15.")

PII_LABELS = ["person", "email", "phone number", "social security number",
              "address", "credit card number", "date"]

result_pii = model.extract_entities(sample, PII_LABELS)
for label, values in result_pii["entities"].items():
    if values:
        print(f"  {label:24s}: {values}")

ents = result_pii["entities"]
assert "John Smith" in ents["person"]
assert ents["email"] == ["jane.doe@acme.com"]
assert ents["credit card number"] == ["4111 1111 1111 1111"]
print("\n PII extraction works.")

## Text classification (detail)

Single-label, multi-label, and custom intent.

In [ ]:
sentiment = model.classify_text(
    "This laptop has amazing performance but terrible battery life!",
    {"sentiment": ["positive", "negative", "neutral"]})
print("sentiment:", sentiment)

aspects = model.classify_text(
    "Great camera quality, decent performance, but poor battery life.",
    {"aspects": {"labels": ["camera", "performance", "battery", "display", "price"],
                 "multi_label": True, "cls_threshold": 0.4}})
print("aspects:  ", aspects)

assert sentiment["sentiment"] == "negative"
assert {"camera", "performance", "battery"}.issubset(set(aspects["aspects"]))
print("\n Classification works.")

## Redaction (optional)

Mask detected PII spans in a column — useful for sharing or storing de-identified text.

In [ ]:
def redact_column(model, df, text_col, pii_labels, batch_size=8, threshold=0.5, new_col="redacted"):
    """Mask detected PII in df[text_col], replacing each span with a [LABEL] placeholder."""
    texts = df[text_col].fillna("").astype(str).tolist()
    results = model.batch_extract_entities(texts, pii_labels, batch_size=batch_size,
                                           threshold=threshold, include_spans=True)
    redacted = []
    for text, res in zip(texts, results):
        spans = [(it["start"], it["end"], label)
                 for label, items in res["entities"].items() for it in items]
        for start, end, label in sorted(spans, key=lambda x: x[0], reverse=True):
            text = text[:start] + f"[{label.replace(' ', '_').upper()}]" + text[end:]
        redacted.append(text)
    out = df.copy()
    out[new_col] = redacted
    return out


red = redact_column(model, df, "customer_note", GIRP_PII_LABELS)
for _, row in red.iterrows():
    print(row["redacted"])

## 6 · Validate on synthetic data (with progress)

Generate a comprehensive synthetic dataset — every tier plus false-positive bait (generic words,
stray numbers) — and classify it **row-by-row with a progress bar**. A quick way to sanity-check
accuracy on your own machine before running real data.

In [ ]:
from synthetic import generate_synthetic_dataset

bench = generate_synthetic_dataset(120, seed=0)        # columns: text, expected
t0 = time.time()
scored = classify_columns(model, bench, ["text"])      # progress bar shows below
dt = time.time() - t0

rank = {l: i for i, l in enumerate(["Public", "Private", "Confidential", "Highly Confidential"])}
acc = (scored["text_girp_level"] == bench["expected"]).mean()
under = sum(rank[p] < rank[e] for p, e in zip(scored["text_girp_level"], bench["expected"]))
fp = int(((scored["text_girp_level"] != "Public") & (bench["expected"] == "Public")).sum())
print(f"\n{len(bench)} rows in {dt:.1f}s (~{len(bench)/dt:.0f} rows/s on {DEVICE})")
print(f"accuracy: {acc*100:.1f}%   under-classifications: {under}   false-positives: {fp}")
print(scored["text_girp_level"].value_counts().to_string())

## Notes & tuning

- **100% local / offline:** the model loads from `model.safetensors` in this folder with the Hub
  disabled — no internet or API calls for inference. Copy the whole folder and it runs air-gapped.
- **Robust by design:** false positives are removed by format validation (a "card" must have 13–19
  digits, an "address" needs a number or street word, pronouns aren't names); structured PII the model
  misses is caught by a Luhn-checked regex backstop (cards, emails, international phones); and CUDA
  out-of-memory recovers automatically (batch size is halved, then CPU fallback). Pass
  `validate=False` to `classify_columns` to see raw model output.
- **Device:** auto CUDA→CPU. On the RTX 2050 it uses fp16 (~0.4 GB VRAM); if VRAM is tight it
  self-recovers, or lower `batch_size`.
- **`threshold`** (default `0.5`): raise to cut false positives, lower to catch more.
- **Tailor it:** edit `PII_LABELS`, or the element sets in `girp.py` (zero-shot — any label works).
- **Validate:** run `python test_girp.py`, or use the synthetic-data cell above on your machine.

## Done
Local offline load · scan chosen DataFrame columns · GIRP sensitivity (4 tiers) · PII · classification · redaction · synthetic validation with progress. Hardened against false positives, missed structured PII, and GPU out-of-memory.